# 03 — Comparação dos modelos e análise de erros

Qwen puro × RAG simples × agente V7

## Objetivo
Ler o placar definitivo gerado por `python app/avaliar_configuracoes.py`. Se o arquivo não estiver presente, apresentar os números **transcritos da última execução compartilhada**, claramente identificados como referência.

In [1]:
from pathlib import Path
import json
RAIZ = Path.cwd()
if not (RAIZ / 'base').exists() and (RAIZ.parent / 'base').exists():
    RAIZ = RAIZ.parent
print('Raiz do projeto:', RAIZ.resolve())


Raiz do projeto: C:\Users\marci\PycharmProjects\assistente-ia-veicular


In [2]:
arq = RAIZ / 'avaliacao' / 'placar_configuracoes.json'
if arq.exists():
    placar = json.loads(arq.read_text(encoding='utf-8'))
    resultados = placar['configuracoes']
    origem = f'Arquivo gerado em {placar.get("data", "data não informada")}'
else:
    resultados = {
        'qwen_puro': {'status_correto_pct': 40.0, 'estrutura_valida_pct': 65.0, 'citacoes_validas_pct': None, 'recusas_corretas_pct': 0.0},
        'rag_simples': {'status_correto_pct': 75.0, 'estrutura_valida_pct': 100.0, 'citacoes_validas_pct': 100.0, 'recusas_corretas_pct': 100.0},
        'rag_agente_v7': {'status_correto_pct': 100.0, 'estrutura_valida_pct': 100.0, 'citacoes_validas_pct': 100.0, 'recusas_corretas_pct': 100.0},
    }
    origem = 'Valores transcritos do terminal compartilhado; arquivo JSON ainda ausente'
print('Origem:', origem)
print(f'{"Configuração":<18} {"Status":>8} {"JSON":>8} {"Citações":>10} {"Recusas":>9}')
for nome, d in resultados.items():
    cit = d.get('citacoes_validas_pct')
    cit = 'N/A' if cit is None else f'{cit:.1f}%'
    print(f'{nome:<18} {d["status_correto_pct"]:>7.1f}% {d["estrutura_valida_pct"]:>7.1f}% {cit:>10} {d["recusas_corretas_pct"]:>8.1f}%')

Origem: Arquivo gerado em 2026-11-17T15:22:09
Configuração         Status     JSON   Citações   Recusas
qwen_puro             40.0%    65.0%        N/A      0.0%
rag_simples           75.0%   100.0%     100.0%    100.0%
rag_agente_v7        100.0%   100.0%     100.0%    100.0%


In [3]:
print('Evolução de status correto:')
print('Qwen puro:   ', resultados['qwen_puro']['status_correto_pct'], '%')
print('RAG simples: ', resultados['rag_simples']['status_correto_pct'], '%')
print('Agente V7:   ', resultados['rag_agente_v7']['status_correto_pct'], '%')
ganho = resultados['rag_agente_v7']['status_correto_pct'] - resultados['rag_simples']['status_correto_pct']
print('Ganho do agente sobre RAG simples:', round(ganho, 2), 'pontos percentuais')

Evolução de status correto:
Qwen puro:    40.0 %
RAG simples:  75.0 %
Agente V7:    100.0 %
Ganho do agente sobre RAG simples: 25.0 pontos percentuais


In [4]:
# Caso existam respostas individuais, verificar diretamente as diferenças de status.
pasta = RAIZ / 'avaliacao'
arquivos = {nome: pasta / f'respostas_{nome}.json' for nome in ('rag_simples','rag_agente_v7')}
if all(arq.exists() for arq in arquivos.values()):
    simples = {str(x['id']): x for x in json.loads(arquivos['rag_simples'].read_text(encoding='utf-8'))}
    agente = {str(x['id']): x for x in json.loads(arquivos['rag_agente_v7'].read_text(encoding='utf-8'))}
    print('Casos com alteração de status entre RAG simples e agente V7:')
    for id_ in sorted(simples, key=int):
        a = simples[id_]['resposta'].get('status')
        b = agente[id_]['resposta'].get('status')
        if a != b: print(f'  {id_}: {a} -> {b}')
else:
    print('Respostas individuais não disponíveis; use os cinco casos documentados abaixo.')

Casos com alteração de status entre RAG simples e agente V7:
  1: esclarecer -> respondido
  6: esclarecer -> respondido
  7: recusa -> respondido
  9: esclarecer -> respondido
  14: recusa -> respondido


## Cinco falhas observadas e correções
| Pergunta | Falha do RAG simples | Tratamento no agente V7 |
|---|---|---|
| 1 — Bateria e painel fraco | Esclarecimento indevido | Política específica para bateria |
| 6 — Pneu no acostamento | Esclarecimento indevido | Política de segurança para pneus |
| 7 — Temperatura alta | Recusa indevida | Regra de superaquecimento |
| 9 — Reservatório quente | Esclarecimento indevido | Orientação explícita para não abrir quente |
| 14 — Onde aguardar socorro | Recusa indevida | Regra para esperar em local protegido |

**Limitações:** 20 perguntas são um conjunto pequeno, e regras foram ajustadas após observar erros nesse mesmo conjunto. Os 100% medem status correto no conjunto conhecido, não generalização. Citações válidas significam URLs/IDs rastreáveis, não revisão humana da fidelidade de cada afirmação. O Qwen puro não tem base documental, portanto citações são N/A.